# Supervised autoencoder on the T = 64 benign + GridSybil windows (benign_gridsybil/T64)

The ConvAutoencoder of `task/encoder` (input 14 × 64 = 13 features + mask, z = 16, 80,189 parameters) with the
classifier head of `task/autoencoder_st` on z (Linear 16→32 → ReLU → Dropout 0.2 → Linear 32→1, 577 parameters;
**80,766** in total). Loss = class-weighted BCE (`pos_weight` = n_benign / n_gridsybil of the **train** windows) + λ ·
masked MSE on real rows (λ = `recon_weight` = 1). Training settings = `task/autoencoder_st` (T48_tvt).

Data: `src/data/encoder_input/benign_gridsybil/T64/` (benign + GridSybil of the two GridSybil folders; on-disk
**90 / 10 train / test by sender vehicle**, no val split). Val = 10% of the train sender vehicles drawn in code with
`task.encoder.data.VehicleSplit(0.1, seed 0)` over all train windows (= the val vehicles of `task/encoder` supervised
mode and of `task/autoencoder_t64`). No split change. Val drives the LR schedule, early stopping, the best checkpoint
and both alarm thresholds (max F1 on val; benign FPR 5% on val). `train.py` never opens the test split; here it is
scored by `task.autoencoder_st_t64.evaluate` (scores first, labels after), which writes `test_results.json`.

Run folder: `AEST64_SEED=<k>` pins `runs/seed<k>` (`npm run aest64:notebook`); otherwise the newest finished run.
Not the thesis TimesNet model.

In [ ]:
import json
import os
from pathlib import Path

assert Path("task/autoencoder_st_t64/config.py").is_file(), "run this notebook from the repo root"

from task.autoencoder_st.metrics import TrackScores, ranking_metrics, threshold_metrics, val_thresholds
from task.autoencoder_st_t64.config import TOTAL_PARAMS, SupervisedAET64Config
from task.autoencoder_st_t64.data import load_test, load_train_val
from task.autoencoder_st_t64.evaluate import evaluate
from task.autoencoder_st_t64.model import SupervisedAutoencoder, count_params
from task.autoencoder_st_t64.trainer import Trainer

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve

settings = SupervisedAET64Config()
if os.environ.get("AEST64_SEED"):
    RUN = Path(settings.runs_dir) / f"seed{int(os.environ['AEST64_SEED'])}"
    assert (RUN / "summary.json").is_file(), f"no finished run in {RUN}"
else:
    runs = sorted(Path(settings.runs_dir).glob("seed*/summary.json"), key=lambda p: p.stat().st_mtime)
    assert runs, "no finished run: npm run aest64:train"
    RUN = runs[-1].parent
summary = json.loads((RUN / "summary.json").read_text())
cfg = SupervisedAET64Config.from_dict(json.loads((RUN / "config.json").read_text()))
META = json.loads((Path(cfg.data_dir) / "metadata.json").read_text())
FEATURES = META["features"]
print("run folder:", RUN, "| seed", cfg.seed, "| data:", cfg.data_dir)

## 1. Input

Train / val as the run used them (val = in-code vehicle draw), test as on disk. No sender vehicle in two splits.
Test labels are not used in this section.

In [ ]:
train, val, label_reads = load_train_val(cfg)
test = load_test(cfg)
veh = {"train": set(train.vehicles), "val": set(val.vehicles), "test": set(test.vehicles)}
assert not (veh["train"] & veh["val"]) and not (veh["train"] & veh["test"]) and not (veh["val"] & veh["test"])
saved_val = set(np.load(RUN / "val_vehicles.npy").tolist())
assert saved_val == veh["val"], "val vehicles differ from the run's val_vehicles.npy"
rows = []
for d in (train, val):
    for cls, v in (("Benign", 0), ("GridSybil", 1)):
        s = d.y == v
        rows.append({"split": d.name, "class": cls, "windows": int(s.sum()), "vehicles": len(set(d.vehicles[s])),
                     "padding share": round(1 - d.mask[s].mean(), 3), "share 1-3 rows": round(float((d.n_messages[s] <= 3).mean()), 3)})
display(pd.DataFrame(rows).set_index(["split", "class"]))
print(f"test: {len(test):,} windows, {len(veh['test']):,} vehicles, padding share {1 - test.mask.mean():.3f}")
print("no vehicle in two splits: True; val vehicles = run's val_vehicles.npy: True; label reads:", label_reads)
pos_weight = (train.y == 0).sum() / (train.y == 1).sum()
assert np.isclose(pos_weight, summary["pos_weight"])
print(f"pos_weight = {pos_weight:.4f} (GridSybil share of train {train.y.mean():.3f})")

## 2. Model

Layers and output shapes for a batch of 2 windows; parameter count asserted (80,766 = 80,189 + 577).

In [ ]:
model = SupervisedAutoencoder(cfg)
x0, m0 = torch.zeros(2, cfg.seq_len, cfg.n_features), torch.ones(2, cfg.seq_len, dtype=torch.bool)
table = []
layer_types = (torch.nn.Conv1d, torch.nn.ConvTranspose1d, torch.nn.Linear, torch.nn.Dropout)
hooks = [mod.register_forward_hook(lambda mod, inp, out, name=name: table.append(
            (name, type(mod).__name__, tuple(out.shape[1:]), sum(p.numel() for p in mod.parameters()))))
         for name, mod in model.named_modules() if isinstance(mod, layer_types)]
model.eval()
with torch.no_grad():
    recon0, z0, logit0 = model(x0, m0)
for h in hooks:
    h.remove()
display(pd.DataFrame(table, columns=["layer", "type", "output (per window)", "params"]))
print(f"trainable parameters: {count_params(model):,} = autoencoder {count_params(model.autoencoder):,} + head {count_params(model.head):,}")
assert count_params(model) == TOTAL_PARAMS == summary["params"] == 80_766

## 3. Training record

Summary (without threshold tables) and the per-epoch history. Train metrics come from the running epoch outputs
(train mode), so they are not strictly comparable with val.

In [ ]:
print(json.dumps({k: v for k, v in summary.items() if k not in ("thresholds", "best_val", "class_counts")}, indent=1))
history = pd.read_json(RUN / "history.jsonl", lines=True)
best = summary["best_epoch"]
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for key, c in (("loss", "tab:blue"), ("bce", "tab:orange"), ("recon", "tab:green")):
    axes[0].plot(history.epoch, history[f"train_{key}"], color=c, ls="--", label=f"train {key}")
    axes[0].plot(history.epoch, history[f"val_{key}"], color=c, label=f"val {key}")
axes[0].set_yscale("log"); axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8, ncol=3)
axes[0].set_title(f"{RUN.name}: per epoch ({summary['stop_reason']}, best epoch {best})")
for key in ("auroc", "pr_auc", "f1", "balanced_accuracy"):
    axes[1].plot(history.epoch, history[f"val_{key}"], label=f"val {key}")
axes[1].plot(history.epoch, history.train_auroc, color="grey", ls="--", label="train auroc (running)")
for a in axes:
    a.axvline(best, color="red", ls="--", lw=1)
axes[1].set_xlabel("epoch"); axes[1].legend(fontsize=8)
plt.show()

The two alarm thresholds, picked on **val** by `train.py`; recomputed here from `best.pt` and compared.

In [ ]:
trainer = Trainer.from_checkpoint(RUN / "best.pt")
val_out = trainer.predict(val.x, val.mask)
thr_win = val_thresholds(val.y, val_out["prob"], cfg.target_fpr)
THR = summary["thresholds"]
same = all(np.isclose(thr_win[k]["threshold"], THR["window"][k]["threshold"], atol=1e-5) for k in thr_win)
print("val AUROC of best.pt:", round(ranking_metrics(val.y, val_out["prob"])["auroc"], 5), "(run:", round(summary["best_val_auroc"], 5), ")")
print("window thresholds equal the run's summary:", same)
pd.DataFrame({k: {m: v[m] for m in ("threshold", "precision", "recall", "f1", "benign_fpr")} for k, v in THR["window"].items()}).T.round(4)

## 4. Test

`evaluate` scores every test window from `best.pt` (x and mask only), then reads labels; writes `test_results.json`
and `test_scores.npz` into the run folder.

In [ ]:
results, raw = evaluate(RUN, test=test)
out, y_win, y_trk = raw["out"], raw["y"], raw["y_trk"]
print(f"windows: {len(test):,}, GridSybil share {y_win.mean():.3f}; tracks {len(raw['tracks']):,}; mixed-label tracks {results['mixed_label_tracks']}")
rows = []
for key, m in results.items():
    if "/" in key:
        level, thr = key.split("/", 1)
        rows.append({"level": level, "threshold": thr, **{k: v for k, v in m.items() if k != "confusion"}})
res = pd.DataFrame(rows).set_index(["level", "threshold"])
res

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
for level, (y, s) in {"window": (y_win, out["prob"]), "track max": (y_trk, raw["track_scores"]["max"])}.items():
    fpr, tpr, _ = roc_curve(y, s)
    p, r, _ = precision_recall_curve(y, s)
    a1.plot(fpr, tpr, label=f"{level} (AUROC {roc_auc_score(y, s):.3f})")
    a2.plot(r, p, label=level)
for name, v in THR["window"].items():
    m = threshold_metrics(y_win, out["prob"], v["threshold"])
    a1.plot(m["benign_fpr"], m["recall"], "o", label=f"window @ {name}")
    a2.plot(m["recall"], m["precision"], "o", label=f"window @ {name}")
a1.plot([0, 1], [0, 1], "k:", lw=1); a1.set_xlabel("benign FPR"); a1.set_ylabel("recall"); a1.legend(fontsize=8)
a2.axhline(y_win.mean(), color="k", ls=":", lw=1); a2.set_xlabel("recall"); a2.set_ylabel("precision"); a2.legend(fontsize=8)
a1.set_title("ROC (test)"); a2.set_title("Precision-recall (test)")
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 51)
ax.hist(out["prob"][y_win == 0], bins=bins, alpha=0.6, density=True, label="benign")
ax.hist(out["prob"][y_win == 1], bins=bins, alpha=0.6, density=True, label="GridSybil")
for (name, v), ls in zip(THR["window"].items(), ("--", ":")):
    ax.axvline(v["threshold"], color="black", ls=ls, lw=1, label=f"val {name} ({v['threshold']:.3f})")
ax.set_yscale("log"); ax.set_xlabel("P(GridSybil)"); ax.legend(fontsize=8); ax.set_title("Test windows by class")
plt.show()

Length shortcut check: AUROC by window length, the length-only AUROC (best direction, picked on test labels as
in the other `autoencoder_st*` notebooks; and the fixed rule −n_messages), per period, and the AUROC of the
reconstruction error.

In [ ]:
print("length-only AUROC (best direction):", results["length_only_auroc"], "| fixed rule -n_messages:", results["length_only_auroc_fixed_neg_n"])
print("reconstruction-error AUROC:", results["reconstruction_error_auroc"])
display(pd.DataFrame(results["by_length"]).T)
display(pd.DataFrame(results["by_period"]).T)
ef = out["error_feature"]
pd.DataFrame({"benign": ef[y_win == 0].mean(0), "GridSybil": ef[y_win == 1].mean(0),
              "AUROC": [roc_auc_score(y_win, ef[:, j]) for j in range(ef.shape[1])]}, index=FEATURES).round(4)

## 5. Context (not a controlled comparison)

Other supervised numbers on GridSybil in this repo, different inputs / splits: `task/autoencoder_st` (T48_tvt,
seeds 0–3, window AUROC 0.9278 ± 0.0019), `task/autoencoder_st_t32` (all/T32_benign_grid, 0.9289 ± 0.0018), and
`task/encoder` supervised mode (T64, encoder + head without decoder; see `task/encoder/runs/evaluation/`).

In [ ]:
print(json.dumps({k: v for k, v in results.items() if not isinstance(v, dict)}, indent=1))
print("window @ val max-F1:", {k: results["window/max_f1"][k] for k in ("auroc", "f1", "recall", "benign_fpr")})
print("window @ val FPR-5%:", {k: results["window/benign_fpr_5pct"][k] for k in ("auroc", "f1", "recall", "benign_fpr")})